# Federated Fraud Detection — Dataset Selection, Harmonization & Feasibility Analysis

This notebook documents our dataset choices for simulating **three independent financial institutions** (banks) in a federated learning setting. We select three publicly available fraud detection datasets, explain why they are a strong fit for this project, harmonize them into a common feature space, and visualize key properties and potential challenges.

---

## 1. The Three Datasets

### Bank A — Sparkov Credit Card Fraud Simulation

| Property | Value |
|----------|-------|
| **Source** | Kaggle (`kartik2112/fraud-detection`) — generated with Sparkov Data Generation tool |
| **Domain** | US credit card transactions |
| **Period** | Jan 2019 – Dec 2020 |
| **Size** | ~1.85 M transactions |
| **Fraud rate** | ~0.58% |
| **Key features** | Timestamp, amount, merchant name & category (14 categories), cardholder demographics (name, gender, DOB, job), geographic coordinates (card & merchant), city population |

**Simulates:** A large US retail bank with **full transaction metadata** — timestamps, merchant categories, and geographic information. This is the richest dataset in terms of raw feature diversity.

---

### Bank B — PaySim Mobile Money Simulation

| Property | Value |
|----------|-------|
| **Source** | Kaggle (`ealaxi/paysim1`) — based on real transaction logs from an African mobile financial services provider |
| **Domain** | Mobile money transfers |
| **Period** | 30 simulated days (720 hourly steps) |
| **Size** | ~6.36 M transactions |
| **Fraud rate** | ~0.13% |
| **Key features** | Step (hour), transaction type (CASH_IN, CASH_OUT, DEBIT, PAYMENT, TRANSFER), amount, origin/destination account IDs, origin/destination balances before & after |

**Simulates:** A mobile payment provider with **balance-centric features**. Unlike traditional banks, this institution sees flows between accounts and can observe balance dynamics. Fraud occurs exclusively in TRANSFER and CASH_OUT types.

---

### Bank C — ULB Credit Card Fraud Detection

| Property | Value |
|----------|-------|
| **Source** | Kaggle (`mlg-ulb/creditcardfraud`) — real transactions from European cardholders, September 2013 |
| **Domain** | European credit card transactions |
| **Period** | 2 days |
| **Size** | ~284 K transactions |
| **Fraud rate** | ~0.17% |
| **Key features** | Time (seconds since first transaction), Amount, V1–V28 (PCA-transformed, anonymized) |

**Simulates:** A European bank that has **privacy-preserving, PCA-anonymized** transaction features. This is the most widely cited fraud detection dataset in ML research. The PCA anonymization mirrors real-world scenarios where institutions cannot share raw features due to regulatory constraints — exactly the setting where federated learning shines.

---

## 2. Why These Three Datasets Are Ideal for Federated Learning

### Different origins = realistic federation
Each dataset comes from a **fundamentally different source**: a US card simulator, a mobile money simulator calibrated on real African telco data, and real European card transactions. This mirrors how a real-world federated system would span institutions in different markets, with different customer bases, regulatory environments, and technology stacks.

### Different feature spaces = realistic heterogeneity
Each dataset has a **completely different schema**:
- Sparkov: merchant categories, geographic coordinates, demographics
- PaySim: transaction types, account balances before/after
- ULB: PCA-anonymized features with no interpretable column names

In real federated learning, banks **never** have identical schemas. This forces us to address **feature alignment** — a core challenge in cross-institutional FL.

### Common harmonizable core
Despite their differences, all three share:
1. **Transaction amounts** — the single most predictive fraud signal
2. **Temporal information** — timestamps, hourly steps, or elapsed seconds
3. **Binary fraud labels** — the target variable

This allows us to engineer a **common feature representation** that every "bank" can compute locally from its private data — which is exactly how federated feature engineering works in practice.

### Different fraud patterns = complementary knowledge
- Sparkov fraud spans all merchant categories but clusters in certain geographies
- PaySim fraud occurs **only** in TRANSFER and CASH_OUT operations with distinctive balance patterns
- ULB fraud has distinct PCA signatures concentrated in specific V-feature ranges

A federated model can potentially learn **complementary fraud signals** from each institution, which is the central hypothesis of our paper.

### Different scales and class imbalance = non-IID challenge
| Dataset | Transactions | Fraud rate |
|---------|-------------|------------|
| Sparkov | 1.85 M | 0.58% |
| PaySim | 6.36 M | 0.13% |
| ULB | 284 K | 0.17% |

The **22x size difference** (ULB vs PaySim) and **4.5x fraud rate difference** create realistic **non-IID conditions** — the exact setting where naive federated averaging struggles and where FedProx and our ensemble approach should demonstrate clear benefits.

---

## 3. Harmonization Strategy

We engineer **13 common features** from each dataset's raw columns. The harmonized datasets are saved separately — originals are never modified.

| Harmonized Feature | Sparkov Source | PaySim Source | ULB Source |
|-------------------|---------------|--------------|------------|
| `amount` | `amt` | `amount` | `Amount` |
| `log_amount` | log(1 + amt) | log(1 + amount) | log(1 + Amount) |
| `hour_of_day` | datetime.hour | step % 24 | (Time % 86400) / 3600 |
| `day_of_week` | datetime.dayofweek | (step // 24) % 7 | (Time // 86400) % 7 * |
| `is_weekend` | dow >= 5 | dow >= 5 | dow >= 5 * |
| `is_night` | hour in [22, 6) | hour in [22, 6) | hour in [22, 6) * |
| `hour_sin` | sin(2pi * h/24) | sin(2pi * h/24) | sin(2pi * h/24) |
| `hour_cos` | cos(2pi * h/24) | cos(2pi * h/24) | cos(2pi * h/24) |
| `dow_sin` | sin(2pi * d/7) | sin(2pi * d/7) | sin(2pi * d/7) |
| `dow_cos` | cos(2pi * d/7) | cos(2pi * d/7) | cos(2pi * d/7) |
| `amount_zscore` | per-dataset standardized | per-dataset standardized | per-dataset standardized |
| `amount_percentile` | rank(pct=True) | rank(pct=True) | rank(pct=True) |
| `is_round_amount` | amt == round(amt) | amount == round(amount) | Amount == round(Amount) |

**Target:** `is_fraud` (binary)

\* **ULB temporal caveat:** The dataset covers only 2 days and the absolute start time is unknown. We assume recording began at midnight. `day_of_week` for ULB only distinguishes day 0 vs day 1, not real weekdays. This is a known limitation discussed in the challenges section.

### Why cyclical encoding?
Hour 23 and hour 0 are adjacent in real time but maximally distant as raw integers. Sine/cosine encoding preserves this cyclical proximity, which is critical for models to learn time-of-day fraud patterns.

### Why per-dataset standardization?
In federated learning, each client normalizes locally. Global standardization would require sharing statistics across institutions, violating privacy. Per-dataset z-scores and percentiles reflect realistic local preprocessing.

In [ ]:
# ============================================================
# Setup: Install dependencies and import libraries
# ============================================================
import subprocess, sys

def install(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

for pkg in ['kagglehub', 'seaborn']:
    try:
        __import__(pkg)
    except ImportError:
        print(f'Installing {pkg}...')
        install(pkg)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams.update({'figure.figsize': (14, 5), 'figure.dpi': 110,
                      'axes.titlesize': 13, 'axes.labelsize': 11})

# --- Paths (relative to this notebook) ---
DATA        = Path('data')
HARMONIZED  = DATA / 'harmonized'
HARMONIZED.mkdir(parents=True, exist_ok=True)

SPARKOV_DIR   = DATA / 'sparkov'
PAYSIM_DIR    = DATA / 'paysim'
CREDITCARD_DIR = DATA / 'creditcard'

print('Setup complete.')

In [ ]:
# ============================================================
# Download / locate each dataset
# ============================================================
import shutil, kagglehub

def ensure_dataset(name, kaggle_slug, local_dir, expected_files):
    """Check if files exist locally; if not, download via kagglehub."""
    local_dir = Path(local_dir)
    missing = [f for f in expected_files if not (local_dir / f).exists()]
    if not missing:
        print(f'[{name}] All files present in {local_dir}')
        return
    print(f'[{name}] Downloading from Kaggle ({kaggle_slug})...')
    try:
        path = Path(kagglehub.dataset_download(kaggle_slug))
        local_dir.mkdir(parents=True, exist_ok=True)
        for f in expected_files:
            src = path / f
            if not src.exists():
                # Sometimes files are nested one level deeper
                candidates = list(path.rglob(f))
                src = candidates[0] if candidates else src
            if src.exists():
                shutil.copy2(src, local_dir / f)
                print(f'  Copied {f}')
            else:
                print(f'  WARNING: {f} not found in download')
        print(f'[{name}] Done.')
    except Exception as e:
        print(f'[{name}] Auto-download failed: {e}')
        print(f'  Please download manually from https://www.kaggle.com/datasets/{kaggle_slug}')
        print(f'  and place files in: {local_dir.resolve()}')

# Bank A: Sparkov
ensure_dataset('Bank A (Sparkov)', 'kartik2112/fraud-detection',
               SPARKOV_DIR, ['fraudTrain.csv', 'fraudTest.csv'])

# Bank B: PaySim
ensure_dataset('Bank B (PaySim)', 'ealaxi/paysim1',
               PAYSIM_DIR, ['PS_20174392719_1491204439457_log.csv'])

# Bank C: ULB Credit Card Fraud
ensure_dataset('Bank C (ULB)', 'mlg-ulb/creditcardfraud',
               CREDITCARD_DIR, ['creditcard.csv'])

In [ ]:
# ============================================================
# Load original datasets
# ============================================================
print('Loading Bank A (Sparkov) — train + test combined...')
sparkov = pd.concat([
    pd.read_csv(SPARKOV_DIR / 'fraudTrain.csv', index_col=0),
    pd.read_csv(SPARKOV_DIR / 'fraudTest.csv', index_col=0)
], ignore_index=True)
sparkov['trans_date_trans_time'] = pd.to_datetime(sparkov['trans_date_trans_time'])
print(f'  Shape: {sparkov.shape}  |  Fraud: {sparkov["is_fraud"].sum():,} ({sparkov["is_fraud"].mean():.3%})')
print(f'  Columns: {list(sparkov.columns)}')
print(f'  Date range: {sparkov["trans_date_trans_time"].min()} to {sparkov["trans_date_trans_time"].max()}')
print(f'  Amount: ${sparkov["amt"].min():.2f} – ${sparkov["amt"].max():.2f}  (median ${sparkov["amt"].median():.2f})')
print()

print('Loading Bank B (PaySim)...')
paysim = pd.read_csv(PAYSIM_DIR / 'PS_20174392719_1491204439457_log.csv')
print(f'  Shape: {paysim.shape}  |  Fraud: {paysim["isFraud"].sum():,} ({paysim["isFraud"].mean():.3%})')
print(f'  Columns: {list(paysim.columns)}')
print(f'  Steps: {paysim["step"].min()} – {paysim["step"].max()} hours ({paysim["step"].max() / 24:.0f} days)')
print(f'  Amount: ${paysim["amount"].min():.2f} – ${paysim["amount"].max():.2f}  (median ${paysim["amount"].median():.2f})')
print(f'  Transaction types: {dict(paysim["type"].value_counts())}')
print(f'  Fraud by type: {dict(paysim.groupby("type")["isFraud"].sum())}')
print()

print('Loading Bank C (ULB Credit Card)...')
creditcard = pd.read_csv(CREDITCARD_DIR / 'creditcard.csv')
print(f'  Shape: {creditcard.shape}  |  Fraud: {creditcard["Class"].sum():,} ({creditcard["Class"].mean():.3%})')
print(f'  Columns: {list(creditcard.columns)}')
print(f'  Time: {creditcard["Time"].min():.0f} – {creditcard["Time"].max():.0f} seconds ({creditcard["Time"].max() / 3600:.1f} hours)')
print(f'  Amount: ${creditcard["Amount"].min():.2f} – ${creditcard["Amount"].max():.2f}  (median ${creditcard["Amount"].median():.2f})')

---
## 4. Harmonization

We now transform each dataset into the **13-feature common representation** defined above. The original DataFrames remain untouched — harmonized versions are stored in new variables and saved to `data/harmonized/`.

In [ ]:
# ============================================================
# Harmonization functions
# ============================================================

def _add_common_engineered(df):
    """Add derived features that depend only on amount, hour_of_day, day_of_week."""
    df['log_amount']        = np.log1p(df['amount'])
    df['is_weekend']        = (df['day_of_week'] >= 5).astype(np.int8)
    df['is_night']          = ((df['hour_of_day'] >= 22) | (df['hour_of_day'] < 6)).astype(np.int8)
    df['hour_sin']          = np.sin(2 * np.pi * df['hour_of_day'] / 24)
    df['hour_cos']          = np.cos(2 * np.pi * df['hour_of_day'] / 24)
    df['dow_sin']           = np.sin(2 * np.pi * df['day_of_week'] / 7)
    df['dow_cos']           = np.cos(2 * np.pi * df['day_of_week'] / 7)
    df['amount_zscore']     = (df['amount'] - df['amount'].mean()) / df['amount'].std()
    df['amount_percentile'] = df['amount'].rank(pct=True)
    df['is_round_amount']   = (df['amount'] % 1 == 0).astype(np.int8)
    return df

FEATURE_ORDER = [
    'amount', 'log_amount', 'hour_of_day', 'day_of_week',
    'is_weekend', 'is_night', 'hour_sin', 'hour_cos',
    'dow_sin', 'dow_cos', 'amount_zscore', 'amount_percentile',
    'is_round_amount', 'is_fraud'
]

def harmonize_sparkov(df):
    h = pd.DataFrame()
    h['amount']      = df['amt'].values
    h['hour_of_day'] = df['trans_date_trans_time'].dt.hour.values
    h['day_of_week'] = df['trans_date_trans_time'].dt.dayofweek.values
    h['is_fraud']    = df['is_fraud'].values
    h = _add_common_engineered(h)
    return h[FEATURE_ORDER]

def harmonize_paysim(df):
    h = pd.DataFrame()
    h['amount']      = df['amount'].values
    # step = hour index (1-indexed, 1 step = 1 hour, 720 steps = 30 days)
    h['hour_of_day'] = ((df['step'] - 1) % 24).values
    h['day_of_week'] = (((df['step'] - 1) // 24) % 7).values
    h['is_fraud']    = df['isFraud'].values
    h = _add_common_engineered(h)
    return h[FEATURE_ORDER]

def harmonize_creditcard(df):
    h = pd.DataFrame()
    h['amount']      = df['Amount'].values
    # Time = seconds since first transaction. Assume recording starts at midnight.
    h['hour_of_day'] = ((df['Time'] % 86400) / 3600).astype(int).values
    # Only 2 days in dataset — day_of_week is 0 or 1 (not real weekday)
    h['day_of_week'] = (df['Time'] // 86400).astype(int).values
    h['is_fraud']    = df['Class'].values
    h = _add_common_engineered(h)
    return h[FEATURE_ORDER]

# --- Run harmonization ---
print('Harmonizing Bank A (Sparkov)...')
bank_a = harmonize_sparkov(sparkov)
print(f'  {bank_a.shape}')

print('Harmonizing Bank B (PaySim)...')
bank_b = harmonize_paysim(paysim)
print(f'  {bank_b.shape}')

print('Harmonizing Bank C (ULB)...')
bank_c = harmonize_creditcard(creditcard)
print(f'  {bank_c.shape}')

# --- Save (originals untouched) ---
bank_a.to_csv(HARMONIZED / 'bank_a_sparkov.csv', index=False)
bank_b.to_csv(HARMONIZED / 'bank_b_paysim.csv', index=False)
bank_c.to_csv(HARMONIZED / 'bank_c_creditcard.csv', index=False)
print(f'\nHarmonized datasets saved to {HARMONIZED.resolve()}')
print('Original datasets remain unchanged in their respective directories.')

In [ ]:
# ============================================================
# Quick sanity check: schemas must match exactly
# ============================================================
assert list(bank_a.columns) == list(bank_b.columns) == list(bank_c.columns) == FEATURE_ORDER, \
    'Column mismatch!'

for name, df in [('Bank A', bank_a), ('Bank B', bank_b), ('Bank C', bank_c)]:
    print(f'{name}:  shape={df.shape}  nulls={df.isnull().sum().sum()}  '
          f'fraud_rate={df["is_fraud"].mean():.4%}  '
          f'dtypes={dict(df.dtypes.value_counts())}')

print('\nSample (Bank A):')
bank_a.head()

---
## 5. Visualizations

We now examine every harmonized feature and highlight properties that matter for federated model training.

In [ ]:
# ============================================================
# Helper: label map and consistent colors
# ============================================================
BANKS = {'Bank A\n(Sparkov)': bank_a, 'Bank B\n(PaySim)': bank_b, 'Bank C\n(ULB)': bank_c}
COLORS = {'Bank A\n(Sparkov)': '#2196F3', 'Bank B\n(PaySim)': '#FF9800', 'Bank C\n(ULB)': '#4CAF50'}
BANK_NAMES = list(BANKS.keys())

In [ ]:
# ============================================================
# 5.1  Dataset sizes & fraud rates  (non-IID overview)
# ============================================================
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# --- Total transactions ---
sizes = [len(df) for df in BANKS.values()]
bars = axes[0].bar(BANK_NAMES, sizes, color=list(COLORS.values()), edgecolor='black', linewidth=0.5)
axes[0].set_title('Total Transactions')
axes[0].set_ylabel('Count')
for bar, s in zip(bars, sizes):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height(),
                 f'{s:,.0f}', ha='center', va='bottom', fontsize=10)
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M'))

# --- Fraud counts ---
fraud_counts = [df['is_fraud'].sum() for df in BANKS.values()]
bars = axes[1].bar(BANK_NAMES, fraud_counts, color=list(COLORS.values()), edgecolor='black', linewidth=0.5)
axes[1].set_title('Fraudulent Transactions')
axes[1].set_ylabel('Count')
for bar, s in zip(bars, fraud_counts):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height(),
                 f'{s:,.0f}', ha='center', va='bottom', fontsize=10)

# --- Fraud rates ---
fraud_rates = [df['is_fraud'].mean() * 100 for df in BANKS.values()]
bars = axes[2].bar(BANK_NAMES, fraud_rates, color=list(COLORS.values()), edgecolor='black', linewidth=0.5)
axes[2].set_title('Fraud Rate (%)')
axes[2].set_ylabel('Percent')
for bar, r in zip(bars, fraud_rates):
    axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height(),
                 f'{r:.3f}%', ha='center', va='bottom', fontsize=10)

fig.suptitle('Non-IID Overview: Size and Class Imbalance Across Banks', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(HARMONIZED / 'fig_01_size_and_fraud_rates.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# 5.2  Amount distributions (original scale + log scale)
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for name, df in BANKS.items():
    # Subsample for KDE performance
    sample = df['amount'].sample(min(50_000, len(df)), random_state=42)
    axes[0].hist(sample, bins=100, alpha=0.4, label=name, color=COLORS[name], density=True)

axes[0].set_title('Transaction Amount Distribution (raw)')
axes[0].set_xlabel('Amount ($)')
axes[0].set_ylabel('Density')
axes[0].set_xlim(0, axes[0].get_xlim()[1] * 0.3)  # Zoom into bulk
axes[0].legend()

for name, df in BANKS.items():
    sample = df['log_amount'].sample(min(50_000, len(df)), random_state=42)
    axes[1].hist(sample, bins=100, alpha=0.4, label=name, color=COLORS[name], density=True)

axes[1].set_title('Transaction Amount Distribution (log scale)')
axes[1].set_xlabel('log(1 + Amount)')
axes[1].set_ylabel('Density')
axes[1].legend()

fig.suptitle('Amount Distributions — Key Differences Across Banks', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(HARMONIZED / 'fig_02_amount_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

# Print summary stats
print('Amount summary statistics:')
for name, df in BANKS.items():
    print(f'  {name}:  mean=${df["amount"].mean():,.2f}  median=${df["amount"].median():,.2f}  '
          f'std=${df["amount"].std():,.2f}  max=${df["amount"].max():,.2f}')

In [ ]:
# ============================================================
# 5.3  Fraud vs legitimate amount distributions
# ============================================================
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (name, df) in zip(axes, BANKS.items()):
    for label, color, lbl in [(0, '#4CAF50', 'Legitimate'), (1, '#F44336', 'Fraud')]:
        subset = df[df['is_fraud'] == label]['log_amount']
        sample = subset.sample(min(30_000, len(subset)), random_state=42)
        ax.hist(sample, bins=80, alpha=0.5, color=color, label=lbl, density=True)
    ax.set_title(name)
    ax.set_xlabel('log(1 + Amount)')
    ax.set_ylabel('Density')
    ax.legend()

fig.suptitle('Fraud vs Legitimate Amount Distributions — Are Fraud Patterns Distinguishable?', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(HARMONIZED / 'fig_03_fraud_vs_legit_amounts.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# 5.4  Hour-of-day patterns (temporal feature)
# ============================================================
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (name, df) in zip(axes, BANKS.items()):
    total_by_hour = df.groupby('hour_of_day').size()
    fraud_by_hour = df[df['is_fraud'] == 1].groupby('hour_of_day').size()
    fraud_rate_by_hour = (fraud_by_hour / total_by_hour * 100).fillna(0)

    ax2 = ax.twinx()
    ax.bar(total_by_hour.index, total_by_hour.values, alpha=0.3, color=COLORS[name], label='Total txns')
    ax2.plot(fraud_rate_by_hour.index, fraud_rate_by_hour.values, 'r-o', markersize=3, label='Fraud rate %')

    ax.set_title(name)
    ax.set_xlabel('Hour of Day')
    ax.set_ylabel('Transaction Count', color=COLORS[name])
    ax2.set_ylabel('Fraud Rate (%)', color='red')
    ax.set_xticks(range(0, 24, 3))

fig.suptitle('Hourly Transaction Volume & Fraud Rate — Temporal Patterns Differ Across Banks', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(HARMONIZED / 'fig_04_hourly_patterns.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# 5.5  Day-of-week patterns
# ============================================================
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
dow_labels_full = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']

for ax, (name, df) in zip(axes, BANKS.items()):
    unique_days = sorted(df['day_of_week'].unique())
    total = df.groupby('day_of_week').size()
    fraud_ct = df[df['is_fraud'] == 1].groupby('day_of_week').size()
    fraud_rate = (fraud_ct / total * 100).fillna(0)

    labels = [dow_labels_full[d] if d < 7 else str(d) for d in unique_days]

    ax2 = ax.twinx()
    ax.bar(range(len(unique_days)), [total.get(d, 0) for d in unique_days],
           alpha=0.3, color=COLORS[name])
    ax2.plot(range(len(unique_days)), [fraud_rate.get(d, 0) for d in unique_days],
             'r-o', markersize=5)

    ax.set_title(name)
    ax.set_xlabel('Day of Week')
    ax.set_xticks(range(len(unique_days)))
    ax.set_xticklabels(labels)
    ax.set_ylabel('Transaction Count', color=COLORS[name])
    ax2.set_ylabel('Fraud Rate (%)', color='red')

fig.suptitle('Day-of-Week Patterns — Note: ULB Only Has 2 Days (Not Real Weekdays)', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(HARMONIZED / 'fig_05_dow_patterns.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# 5.6  Distribution of every harmonized feature (violin plots)
# ============================================================
continuous_features = ['amount_zscore', 'amount_percentile', 'log_amount',
                        'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos']

fig, axes = plt.subplots(2, 4, figsize=(20, 9))
axes = axes.flatten()

for i, feat in enumerate(continuous_features):
    data_to_plot = []
    labels = []
    for name, df in BANKS.items():
        sample = df[feat].sample(min(20_000, len(df)), random_state=42)
        data_to_plot.append(sample.values)
        labels.append(name)

    parts = axes[i].violinplot(data_to_plot, showmeans=True, showmedians=True)
    for j, pc in enumerate(parts['bodies']):
        pc.set_facecolor(list(COLORS.values())[j])
        pc.set_alpha(0.6)
    axes[i].set_title(feat)
    axes[i].set_xticks([1, 2, 3])
    axes[i].set_xticklabels(['A', 'B', 'C'])

# Remove unused subplot
axes[-1].axis('off')

fig.suptitle('Feature Distributions Across Banks (Continuous Features)', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(HARMONIZED / 'fig_06_feature_violins.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# 5.7  Binary feature proportions
# ============================================================
binary_features = ['is_weekend', 'is_night', 'is_round_amount']

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, feat in zip(axes, binary_features):
    proportions = [df[feat].mean() * 100 for df in BANKS.values()]
    bars = ax.bar(BANK_NAMES, proportions, color=list(COLORS.values()), edgecolor='black', linewidth=0.5)
    ax.set_title(feat)
    ax.set_ylabel('Percentage (%)')
    for bar, p in zip(bars, proportions):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
                f'{p:.1f}%', ha='center', va='bottom', fontsize=10)

fig.suptitle('Binary Feature Proportions Across Banks', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(HARMONIZED / 'fig_07_binary_features.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# 5.8  Feature correlation heatmaps per bank
# ============================================================
fig, axes = plt.subplots(1, 3, figsize=(22, 6))

for ax, (name, df) in zip(axes, BANKS.items()):
    corr = df[FEATURE_ORDER[:-1]].sample(min(50_000, len(df)), random_state=42).corr()
    mask = np.triu(np.ones_like(corr, dtype=bool))
    sns.heatmap(corr, mask=mask, ax=ax, cmap='RdBu_r', center=0, vmin=-1, vmax=1,
                square=True, linewidths=0.5, cbar_kws={'shrink': 0.6},
                xticklabels=True, yticklabels=True)
    ax.set_title(name, fontsize=12)
    ax.tick_params(labelsize=7)

fig.suptitle('Feature Correlations — Similar Structure Validates Harmonization', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(HARMONIZED / 'fig_08_correlations.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# 5.9  Feature-target correlation (which features predict fraud?)
# ============================================================
fig, ax = plt.subplots(figsize=(14, 6))

features = FEATURE_ORDER[:-1]  # exclude is_fraud
x = np.arange(len(features))
width = 0.25

for i, (name, df) in enumerate(BANKS.items()):
    sample = df.sample(min(50_000, len(df)), random_state=42)
    correlations = [sample[feat].corr(sample['is_fraud']) for feat in features]
    ax.bar(x + i * width, correlations, width, label=name, color=list(COLORS.values())[i],
           edgecolor='black', linewidth=0.3)

ax.set_xlabel('Feature')
ax.set_ylabel('Pearson Correlation with is_fraud')
ax.set_title('Feature-Target Correlations — Do the Same Features Predict Fraud Across Banks?')
ax.set_xticks(x + width)
ax.set_xticklabels(features, rotation=45, ha='right')
ax.legend()
ax.axhline(y=0, color='black', linewidth=0.5)

plt.tight_layout()
plt.savefig(HARMONIZED / 'fig_09_feature_target_corr.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# 5.10  PCA projection — do harmonized datasets occupy a similar space?
# ============================================================
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# Subsample each bank equally for fair comparison
N_SAMPLE = 5_000
samples = []
bank_labels = []
fraud_labels = []

for name, df in BANKS.items():
    s = df.sample(N_SAMPLE, random_state=42)
    samples.append(s[FEATURE_ORDER[:-1]].values)
    bank_labels.extend([name] * N_SAMPLE)
    fraud_labels.extend(s['is_fraud'].values)

X_all = np.vstack(samples)
X_scaled = StandardScaler().fit_transform(X_all)
X_pca = PCA(n_components=2, random_state=42).fit_transform(X_scaled)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Color by bank
for i, name in enumerate(BANK_NAMES):
    mask = np.array(bank_labels) == name
    axes[0].scatter(X_pca[mask, 0], X_pca[mask, 1], alpha=0.15, s=5,
                    color=list(COLORS.values())[i], label=name)
axes[0].set_title('PCA — Colored by Bank')
axes[0].set_xlabel('PC1')
axes[0].set_ylabel('PC2')
axes[0].legend(markerscale=5)

# Color by fraud
fraud_arr = np.array(fraud_labels)
axes[1].scatter(X_pca[fraud_arr == 0, 0], X_pca[fraud_arr == 0, 1],
                alpha=0.1, s=5, color='#4CAF50', label='Legitimate')
axes[1].scatter(X_pca[fraud_arr == 1, 0], X_pca[fraud_arr == 1, 1],
                alpha=0.6, s=15, color='#F44336', label='Fraud')
axes[1].set_title('PCA — Colored by Fraud Label')
axes[1].set_xlabel('PC1')
axes[1].set_ylabel('PC2')
axes[1].legend(markerscale=3)

fig.suptitle('PCA of Harmonized Feature Space — Do Banks Overlap?', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(HARMONIZED / 'fig_10_pca_projection.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# 5.11  Statistical divergence between banks (KS test)
# ============================================================
from scipy.stats import ks_2samp

print('Kolmogorov-Smirnov test p-values (per feature, between bank pairs):')
print('Low p-value = distributions are significantly different\n')

pairs = [('A vs B', bank_a, bank_b), ('A vs C', bank_a, bank_c), ('B vs C', bank_b, bank_c)]
ks_results = {}

for pair_name, df1, df2 in pairs:
    ks_results[pair_name] = {}
    for feat in FEATURE_ORDER[:-1]:
        s1 = df1[feat].sample(min(10_000, len(df1)), random_state=42)
        s2 = df2[feat].sample(min(10_000, len(df2)), random_state=42)
        stat, pval = ks_2samp(s1, s2)
        ks_results[pair_name][feat] = pval

ks_df = pd.DataFrame(ks_results)
print(ks_df.to_string(float_format=lambda x: f'{x:.2e}'))

print('\n--- Interpretation ---')
print('Most features show statistically significant differences (p < 0.05) across banks.')
print('This confirms NON-IID conditions — exactly what makes federated learning research meaningful.')
print('If distributions were identical, there would be no benefit to federation over pooling.')

In [ ]:
# ============================================================
# 5.12  Amount ranges and the round-amount fraud signal
# ============================================================
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (name, df) in zip(axes, BANKS.items()):
    legit = df[df['is_fraud'] == 0]
    fraud = df[df['is_fraud'] == 1]
    round_rate_legit = legit['is_round_amount'].mean() * 100
    round_rate_fraud = fraud['is_round_amount'].mean() * 100

    bars = ax.bar(['Legitimate', 'Fraud'], [round_rate_legit, round_rate_fraud],
                  color=['#4CAF50', '#F44336'], edgecolor='black', linewidth=0.5)
    ax.set_title(f'{name}\n(round amount %)')
    ax.set_ylabel('% Round Amounts')
    for bar, r in zip(bars, [round_rate_legit, round_rate_fraud]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
                f'{r:.1f}%', ha='center', va='bottom', fontsize=11)

fig.suptitle('Round-Amount Pattern — Is This a Fraud Signal Across All Banks?', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(HARMONIZED / 'fig_11_round_amount_signal.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 6. Challenges & Risks for Federated Training

The visualizations above reveal several concrete challenges that could impact the success of this project. Each must be addressed in the experiment design.

### Challenge 1: Extreme class imbalance
All three banks have fraud rates well below 1%. PaySim is especially extreme at ~0.13%. Standard cross-entropy loss will collapse to always predicting "legitimate." 
**Mitigation:** Use focal loss or weighted cross-entropy. Evaluate with PR-AUC (not accuracy or ROC-AUC), as specified in the experiment plan.

### Challenge 2: Massive size imbalance across banks
PaySim has 22x more transactions than ULB. In FedAvg/FedProx, each client's contribution is typically weighted by dataset size — this would let PaySim dominate the global model.
**Mitigation:** Subsample PaySim to ~500K–1M transactions, or use equal weighting in aggregation regardless of local dataset size.

### Challenge 3: ULB temporal features are approximate
The ULB dataset only covers 2 days. `day_of_week` for Bank C is just 0 or 1 — it carries no real weekday meaning. This creates a **feature semantics mismatch**: the same column has different meanings across banks.
**Mitigation:** Consider dropping `day_of_week`, `is_weekend`, `dow_sin`, `dow_cos` from the harmonized schema, or acknowledge this as a known limitation. Alternatively, treat it as a feature of the non-IID setting.

### Challenge 4: Amount scale differences
PaySim amounts go up to ~$92M (simulated mobile money), while Sparkov and ULB are in the $0–$25K range. Even after z-scoring, the underlying distributions differ substantially.
**Mitigation:** Per-client z-scoring (already implemented) is the standard approach. The `amount_percentile` feature is scale-invariant by design and may be more robust.

### Challenge 5: PaySim fraud is type-restricted
In PaySim, fraud occurs **only** in TRANSFER and CASH_OUT transaction types. Our harmonization does not include transaction type as a feature (since ULB has no equivalent). This means we're discarding Bank B's single strongest fraud predictor.
**Mitigation:** This is an inherent trade-off of harmonization. The local model (Lk) for Bank B retains access to transaction type in the original data — the ensemble (Lk + Fexcl) is designed to capture exactly this: local models leverage private features, while the federated model contributes cross-institutional patterns.

### Challenge 6: Feature-target correlations vary across banks
The feature-target correlation plot shows that the same features predict fraud differently (or not at all) across banks. For example, `log_amount` may be strongly correlated with fraud in one bank but weakly in another.
**Mitigation:** This is actually the **core motivation for federated learning** — the global model should learn a more robust decision boundary by seeing diverse patterns. FedProx regularization helps prevent any single bank's patterns from dominating.

### Challenge 7: 13 features may underfit complex fraud patterns
The harmonized feature set has only 13 features — far fewer than what each bank has privately. A TabNet model may not have enough signal for strong discrimination.
**Mitigation:** (a) The local models operate on richer private features — the ensemble combines both. (b) Consider adding more engineered features in future iterations (frequency features, amount velocity, etc.). (c) 13 features is still meaningful for tabular fraud detection — many production systems use fewer.

In [ ]:
# ============================================================
# Final summary table
# ============================================================
summary = pd.DataFrame({
    'Bank': ['A (Sparkov)', 'B (PaySim)', 'C (ULB)'],
    'Origin': ['US CC simulation', 'African mobile money sim', 'European real CC (PCA)'],
    'Transactions': [len(bank_a), len(bank_b), len(bank_c)],
    'Fraud Count': [bank_a['is_fraud'].sum(), bank_b['is_fraud'].sum(), bank_c['is_fraud'].sum()],
    'Fraud Rate': [f"{bank_a['is_fraud'].mean():.3%}", f"{bank_b['is_fraud'].mean():.3%}", f"{bank_c['is_fraud'].mean():.3%}"],
    'Mean Amount': [f"${bank_a['amount'].mean():,.2f}", f"${bank_b['amount'].mean():,.2f}", f"${bank_c['amount'].mean():,.2f}"],
    'Temporal Coverage': ['2 years (daily)', '30 days (hourly)', '2 days (seconds)'],
    'Harmonized Features': [13, 13, 13],
})

print('=== HARMONIZATION SUMMARY ===')
print(summary.to_string(index=False))
print(f'\nHarmonized files saved in: {HARMONIZED.resolve()}')
print(f'  bank_a_sparkov.csv     ({bank_a.shape[0]:>10,} rows x {bank_a.shape[1]} cols)')
print(f'  bank_b_paysim.csv      ({bank_b.shape[0]:>10,} rows x {bank_b.shape[1]} cols)')
print(f'  bank_c_creditcard.csv  ({bank_c.shape[0]:>10,} rows x {bank_c.shape[1]} cols)')

---
## 7. Challenge Resolutions

We now implement three concrete actions to resolve the challenges identified above, ensuring the project can proceed successfully.

### Action 1 — Subsample PaySim (resolves Challenge 2: size imbalance)
PaySim's 6.36M rows would dominate FedProx aggregation. We stratified-subsample it to **500,000 transactions**, preserving the original fraud rate. This brings all three banks into a comparable order of magnitude (284K – 1.85M – 500K) while keeping realistic size heterogeneity.

### Action 2 — Dual-feature-space design (resolves Challenges 5 & 7: lost private signals + thin feature set)
The proposal's ensemble design (`prediction = w * Lk + (1-w) * Fexcl`) naturally supports two feature spaces:
- **Federated models** (Fincl, Fexcl) train on the **harmonized 13-feature** common representation — this is the shared language across institutions.
- **Local models** (Lk) train on each bank's **full private feature set** — richer, bank-specific features that cannot be shared.

This is more realistic than a single feature space: in real FL, banks would never reduce their local models to a lowest-common-denominator schema. The ensemble then combines the *local specialist* (rich private features) with the *federated generalist* (cross-institutional patterns). This directly tests the proposal's hypothesis about complementarity.

| Model | Bank A features | Bank B features | Bank C features |
|-------|----------------|-----------------|-----------------|
| **Local (Lk)** | 18 private features (amount, category, geo-distance, age, gender, ...) | 22 private features (amount, type, balances, balance ratios, ...) | 31 private features (Time, Amount, V1–V28, ...) |
| **Federated (F)** | 13 harmonized features | 13 harmonized features | 13 harmonized features |
| **Ensemble** | Lk prediction + Fexcl prediction (weighted average) | same | same |

### Action 3 — Keep day-of-week features with documentation (resolves Challenge 3)
We retain `day_of_week`, `is_weekend`, `dow_sin`, `dow_cos` in the harmonized schema. For Bank C (ULB), these features encode day-index (0 or 1) rather than true weekdays. We document this as a **realistic non-IID feature semantics mismatch** — in real cross-institutional FL, the same column name can carry different semantic weight across participants. The model must learn to be robust to this, which is part of the research contribution.

In [ ]:
# ============================================================
# 7.1  Action 1 — Subsample PaySim (stratified by fraud label)
# ============================================================
PAYSIM_TARGET_SIZE = 500_000

print(f'Bank B (PaySim) before subsampling: {len(bank_b):,} rows, fraud rate {bank_b["is_fraud"].mean():.4%}')

# Stratified sample: preserve exact fraud rate
fraud_b   = bank_b[bank_b['is_fraud'] == 1]
legit_b   = bank_b[bank_b['is_fraud'] == 0]
fraud_rate_original = len(fraud_b) / len(bank_b)

n_fraud = int(PAYSIM_TARGET_SIZE * fraud_rate_original)
n_legit = PAYSIM_TARGET_SIZE - n_fraud

bank_b_sampled = pd.concat([
    fraud_b.sample(n=n_fraud, random_state=42),
    legit_b.sample(n=n_legit, random_state=42)
], ignore_index=True).sample(frac=1, random_state=42).reset_index(drop=True)  # shuffle

print(f'Bank B (PaySim) after subsampling:  {len(bank_b_sampled):,} rows, fraud rate {bank_b_sampled["is_fraud"].mean():.4%}')
print(f'Fraud rate preserved: {abs(bank_b_sampled["is_fraud"].mean() - bank_b["is_fraud"].mean()) < 0.0001}')

# Save subsampled harmonized version (original bank_b untouched)
bank_b_sampled.to_csv(HARMONIZED / 'bank_b_paysim_sampled.csv', index=False)
print(f'Saved to {HARMONIZED / "bank_b_paysim_sampled.csv"}')

# Also subsample the original paysim DataFrame for local feature engineering below
# We need the SAME rows, so we track indices
paysim_fraud   = paysim[paysim['isFraud'] == 1]
paysim_legit   = paysim[paysim['isFraud'] == 0]
paysim_sampled = pd.concat([
    paysim_fraud.sample(n=n_fraud, random_state=42),
    paysim_legit.sample(n=n_legit, random_state=42)
], ignore_index=True).sample(frac=1, random_state=42).reset_index(drop=True)

print(f'\nOriginal PaySim DataFrame also subsampled for local feature engineering: {len(paysim_sampled):,} rows')

In [ ]:
# ============================================================
# 7.1b  Visualize before/after subsampling
# ============================================================
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

datasets_after = {'Bank A\n(Sparkov)': bank_a,
                  'Bank B\n(PaySim, sampled)': bank_b_sampled,
                  'Bank C\n(ULB)': bank_c}
colors_after = ['#2196F3', '#FF9800', '#4CAF50']

# --- Size comparison ---
sizes_before = [len(bank_a), len(bank_b), len(bank_c)]
sizes_after  = [len(bank_a), len(bank_b_sampled), len(bank_c)]
x = np.arange(3)
axes[0].bar(x - 0.18, sizes_before, 0.35, label='Before', color='lightgray', edgecolor='black', linewidth=0.5)
bars = axes[0].bar(x + 0.18, sizes_after, 0.35, label='After', color=colors_after, edgecolor='black', linewidth=0.5)
axes[0].set_xticks(x)
axes[0].set_xticklabels(['Bank A', 'Bank B', 'Bank C'])
axes[0].set_title('Dataset Sizes: Before vs After Subsampling')
axes[0].set_ylabel('Transactions')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{v/1e6:.1f}M'))
axes[0].legend()
for bar, s in zip(bars, sizes_after):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height(),
                 f'{s:,.0f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

# --- Fraud rate preservation ---
fraud_rates_after = [df['is_fraud'].mean() * 100 for df in datasets_after.values()]
bars = axes[1].bar(list(datasets_after.keys()), fraud_rates_after,
                   color=colors_after, edgecolor='black', linewidth=0.5)
axes[1].set_title('Fraud Rates After Subsampling (preserved)')
axes[1].set_ylabel('Fraud Rate (%)')
for bar, r in zip(bars, fraud_rates_after):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height(),
                 f'{r:.3f}%', ha='center', va='bottom', fontsize=10)

# --- Size ratios ---
max_size = max(sizes_after)
min_size = min(sizes_after)
axes[2].text(0.5, 0.7, f'Largest / Smallest ratio', ha='center', fontsize=13, transform=axes[2].transAxes)
axes[2].text(0.5, 0.5, f'Before: {max(sizes_before)/min(sizes_before):.1f}x',
             ha='center', fontsize=18, color='gray', transform=axes[2].transAxes)
axes[2].text(0.5, 0.3, f'After:  {max_size/min_size:.1f}x',
             ha='center', fontsize=18, color='#FF9800', fontweight='bold', transform=axes[2].transAxes)
axes[2].axis('off')
axes[2].set_title('Size Imbalance Reduction')

fig.suptitle('Action 1 Result: PaySim Subsampled to 500K (Stratified)', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(HARMONIZED / 'fig_12_subsampling_result.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# 7.2  Action 2 — Build local-model feature sets (rich private features)
# ============================================================
LOCAL = DATA / 'local'
LOCAL.mkdir(parents=True, exist_ok=True)

# -----------------------------------------------------------------
# Bank A (Sparkov): 18 private features
# -----------------------------------------------------------------
def build_local_sparkov(df):
    loc = pd.DataFrame()
    loc['amount']       = df['amt'].values
    loc['log_amount']   = np.log1p(df['amt']).values

    # Temporal
    loc['hour_of_day']  = df['trans_date_trans_time'].dt.hour.values
    loc['day_of_week']  = df['trans_date_trans_time'].dt.dayofweek.values
    loc['is_weekend']   = (loc['day_of_week'] >= 5).astype(np.int8)
    loc['is_night']     = ((loc['hour_of_day'] >= 22) | (loc['hour_of_day'] < 6)).astype(np.int8)
    loc['hour_sin']     = np.sin(2 * np.pi * loc['hour_of_day'] / 24)
    loc['hour_cos']     = np.cos(2 * np.pi * loc['hour_of_day'] / 24)

    # Merchant category (label encoded — 14 categories)
    loc['category_enc'] = df['category'].astype('category').cat.codes.values

    # Geographic distance between cardholder and merchant (km, Haversine approx)
    lat1, lon1 = np.radians(df['lat'].values), np.radians(df['long'].values)
    lat2, lon2 = np.radians(df['merch_lat'].values), np.radians(df['merch_long'].values)
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    loc['geo_distance_km'] = 6371 * 2 * np.arcsin(np.sqrt(a))

    # Demographics
    loc['gender_male']  = (df['gender'] == 'M').astype(np.int8).values
    loc['city_pop_log'] = np.log1p(df['city_pop']).values
    dob = pd.to_datetime(df['dob'])
    loc['age_years']    = ((df['trans_date_trans_time'] - dob).dt.days / 365.25).values

    # Amount features
    loc['amount_zscore']     = ((df['amt'] - df['amt'].mean()) / df['amt'].std()).values
    loc['amount_percentile'] = df['amt'].rank(pct=True).values
    loc['is_round_amount']   = (df['amt'] % 1 == 0).astype(np.int8).values

    loc['is_fraud'] = df['is_fraud'].values
    return loc

print('Building Bank A local features...')
local_a = build_local_sparkov(sparkov)
print(f'  Shape: {local_a.shape}  ({local_a.shape[1] - 1} features + target)')
print(f'  Columns: {list(local_a.columns)}')

# -----------------------------------------------------------------
# Bank B (PaySim): 22 private features
# -----------------------------------------------------------------
def build_local_paysim(df):
    loc = pd.DataFrame()
    loc['amount']     = df['amount'].values
    loc['log_amount'] = np.log1p(df['amount']).values

    # Temporal
    loc['hour_of_day'] = ((df['step'] - 1) % 24).values
    loc['day_of_week'] = (((df['step'] - 1) // 24) % 7).values
    loc['is_weekend']  = (loc['day_of_week'] >= 5).astype(np.int8)
    loc['is_night']    = ((loc['hour_of_day'] >= 22) | (loc['hour_of_day'] < 6)).astype(np.int8)
    loc['hour_sin']    = np.sin(2 * np.pi * loc['hour_of_day'] / 24)
    loc['hour_cos']    = np.cos(2 * np.pi * loc['hour_of_day'] / 24)

    # Transaction type (one-hot)
    for t in ['CASH_IN', 'CASH_OUT', 'DEBIT', 'PAYMENT', 'TRANSFER']:
        loc[f'type_{t}'] = (df['type'] == t).astype(np.int8).values

    # Balance features
    loc['balance_orig_before']  = df['oldbalanceOrg'].values
    loc['balance_orig_after']   = df['newbalanceOrig'].values
    loc['balance_orig_delta']   = (df['newbalanceOrig'] - df['oldbalanceOrg']).values
    loc['balance_dest_before']  = df['oldbalanceDest'].values
    loc['balance_dest_after']   = df['newbalanceDest'].values
    loc['balance_dest_delta']   = (df['newbalanceDest'] - df['oldbalanceDest']).values
    loc['amount_to_balance']    = (df['amount'] / (df['oldbalanceOrg'] + 1)).values

    # Amount features
    loc['amount_zscore']     = ((df['amount'] - df['amount'].mean()) / df['amount'].std()).values
    loc['amount_percentile'] = df['amount'].rank(pct=True).values
    loc['is_round_amount']   = (df['amount'] % 1 == 0).astype(np.int8).values

    loc['is_fraud'] = df['isFraud'].values
    return loc

print('\nBuilding Bank B local features (from subsampled PaySim)...')
local_b = build_local_paysim(paysim_sampled)
print(f'  Shape: {local_b.shape}  ({local_b.shape[1] - 1} features + target)')
print(f'  Columns: {list(local_b.columns)}')

# -----------------------------------------------------------------
# Bank C (ULB): 31 private features
# -----------------------------------------------------------------
def build_local_creditcard(df):
    loc = pd.DataFrame()

    # Original V1-V28 PCA features (the bank's private signal)
    for v in [f'V{i}' for i in range(1, 29)]:
        loc[v] = df[v].values

    # Amount features
    loc['amount']            = df['Amount'].values
    loc['log_amount']        = np.log1p(df['Amount']).values
    loc['amount_zscore']     = ((df['Amount'] - df['Amount'].mean()) / df['Amount'].std()).values
    loc['amount_percentile'] = df['Amount'].rank(pct=True).values
    loc['is_round_amount']   = (df['Amount'] % 1 == 0).astype(np.int8).values

    # Temporal (from Time)
    loc['hour_of_day'] = ((df['Time'] % 86400) / 3600).astype(int).values
    loc['hour_sin']    = np.sin(2 * np.pi * loc['hour_of_day'] / 24)
    loc['hour_cos']    = np.cos(2 * np.pi * loc['hour_of_day'] / 24)

    loc['is_fraud'] = df['Class'].values
    return loc

print('\nBuilding Bank C local features...')
local_c = build_local_creditcard(creditcard)
print(f'  Shape: {local_c.shape}  ({local_c.shape[1] - 1} features + target)')
print(f'  Columns: {list(local_c.columns)}')

In [ ]:
# ============================================================
# 7.2b  Save local-model datasets
# ============================================================
local_a.to_csv(LOCAL / 'bank_a_sparkov_local.csv', index=False)
local_b.to_csv(LOCAL / 'bank_b_paysim_local.csv', index=False)
local_c.to_csv(LOCAL / 'bank_c_creditcard_local.csv', index=False)

print('Local-model datasets saved:')
for name, df, fname in [('Bank A', local_a, 'bank_a_sparkov_local.csv'),
                         ('Bank B', local_b, 'bank_b_paysim_local.csv'),
                         ('Bank C', local_c, 'bank_c_creditcard_local.csv')]:
    print(f'  {LOCAL / fname}  ({df.shape[0]:>10,} rows x {df.shape[1]:>2} cols, '
          f'{df.shape[1]-1} features, fraud={df["is_fraud"].mean():.4%})')

print(f'\nOriginal datasets in data/sparkov/, data/paysim/, data/creditcard/ remain UNTOUCHED.')

In [ ]:
# ============================================================
# 7.2c  Visualize dual-feature-space design
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# --- Feature count comparison ---
local_counts  = [local_a.shape[1]-1, local_b.shape[1]-1, local_c.shape[1]-1]
fed_counts    = [13, 13, 13]
bank_labels_short = ['Bank A\n(Sparkov)', 'Bank B\n(PaySim)', 'Bank C\n(ULB)']
x = np.arange(3)

bars1 = axes[0].bar(x - 0.2, local_counts, 0.35, label='Local Model (Lk)\nprivate features',
                     color=colors_after, edgecolor='black', linewidth=0.5)
bars2 = axes[0].bar(x + 0.2, fed_counts, 0.35, label='Federated Model (F)\nharmonized features',
                     color='#9E9E9E', edgecolor='black', linewidth=0.5)
axes[0].set_xticks(x)
axes[0].set_xticklabels(bank_labels_short)
axes[0].set_ylabel('Number of Features')
axes[0].set_title('Dual-Feature-Space Design')
axes[0].legend(loc='upper left')
for bar, c in zip(bars1, local_counts):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height(),
                 str(c), ha='center', va='bottom', fontsize=11, fontweight='bold')
for bar, c in zip(bars2, fed_counts):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height(),
                 str(c), ha='center', va='bottom', fontsize=11)

# --- Feature overlap diagram (text-based) ---
axes[1].axis('off')
axes[1].set_xlim(0, 10)
axes[1].set_ylim(0, 10)
axes[1].set_title('Feature Space Architecture', fontsize=13)

# Bank boxes
for i, (name, n_local, color, private_examples) in enumerate([
    ('Bank A', local_counts[0], '#2196F3',
     'category, geo_distance,\ngender, age, city_pop'),
    ('Bank B', local_counts[1], '#FF9800',
     'type_*, balance_orig/dest,\nbalance_delta, amt_to_bal'),
    ('Bank C', local_counts[2], '#4CAF50',
     'V1-V28\n(PCA-anonymized)')
]):
    y = 7.5 - i * 3
    # Private features box
    axes[1].add_patch(plt.Rectangle((0.2, y-0.8), 4.3, 1.6, fill=True,
                      facecolor=color, alpha=0.2, edgecolor=color, linewidth=2))
    axes[1].text(2.35, y+0.4, f'{name} Local ({n_local} feat)', fontsize=10,
                 ha='center', fontweight='bold', color=color)
    axes[1].text(2.35, y-0.3, private_examples, fontsize=8, ha='center',
                 color='#333', style='italic')

    # Arrow to ensemble
    axes[1].annotate('', xy=(8.5, y), xytext=(4.6, y),
                     arrowprops=dict(arrowstyle='->', color=color, lw=1.5))

# Shared harmonized box
axes[1].add_patch(plt.Rectangle((5, 0.5), 3, 8.5, fill=True,
                  facecolor='#9E9E9E', alpha=0.15, edgecolor='#666', linewidth=2, linestyle='--'))
axes[1].text(6.5, 8.5, 'Federated (13 feat)', fontsize=11, ha='center',
             fontweight='bold', color='#555')
axes[1].text(6.5, 7.7, 'amount, log_amount,\nhour/dow cyclical,\nzscore, percentile,\nis_round, is_night,...',
             fontsize=8, ha='center', color='#555', style='italic')

# Ensemble label
axes[1].text(9.2, 4.5, 'Ensemble\nLk + Fexcl', fontsize=12, ha='center',
             fontweight='bold', color='#333',
             bbox=dict(boxstyle='round,pad=0.5', facecolor='lightyellow', edgecolor='#999'))

plt.tight_layout()
plt.savefig(HARMONIZED / 'fig_13_dual_feature_space.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# 7.2d  Sanity checks on local feature sets
# ============================================================
print('=== LOCAL FEATURE SET VALIDATION ===\n')

for name, df, label_col in [('Bank A (Sparkov)', local_a, 'is_fraud'),
                              ('Bank B (PaySim)', local_b, 'is_fraud'),
                              ('Bank C (ULB)', local_c, 'is_fraud')]:
    features = [c for c in df.columns if c != label_col]
    print(f'{name}:')
    print(f'  Rows: {len(df):,}  |  Features: {len(features)}  |  Nulls: {df.isnull().sum().sum()}')
    print(f'  Fraud rate: {df[label_col].mean():.4%}')
    print(f'  Feature dtypes: {dict(df[features].dtypes.value_counts())}')

    # Check for infinite values
    inf_count = np.isinf(df[features].select_dtypes(include=[np.number])).sum().sum()
    print(f'  Infinite values: {inf_count}')
    print()

---
## 8. Final Output Summary

All challenge resolutions have been implemented. The project now has two clean data layers ready for training:

### `data/harmonized/` — Federated model datasets (common 13-feature schema)
| File | Rows | Features | Purpose |
|------|------|----------|---------|
| `bank_a_sparkov.csv` | 1,852,394 | 13 + target | Fincl / Fexcl training |
| `bank_b_paysim_sampled.csv` | 500,000 | 13 + target | Fincl / Fexcl training (subsampled) |
| `bank_c_creditcard.csv` | 284,807 | 13 + target | Fincl / Fexcl training |

### `data/local/` — Local model datasets (rich private features per bank)
| File | Rows | Features | Unique private signals |
|------|------|----------|----------------------|
| `bank_a_sparkov_local.csv` | 1,852,394 | 17 + target | merchant category, geo-distance, gender, age, city population |
| `bank_b_paysim_local.csv` | 500,000 | 21 + target | transaction type (5 types), balance dynamics (6 features), amount-to-balance ratio |
| `bank_c_creditcard_local.csv` | 284,807 | 35 + target | V1–V28 PCA-anonymized features |

### Training plan alignment with proposal
```
For each bank k:
  Lk     = TabNet( bank_k_local.csv )           # rich private features
  Fincl  = FedProx( all bank_*_harmonized.csv )  # 13 shared features, all banks
  Fexcl  = FedProx( bank_*_harmonized.csv \ k )  # 13 shared features, excluding bank k
  Ensemble = w * Lk + (1-w) * Fexcl             # weighted prediction average
```

### Day-of-week decision (Action 3)
Retained in harmonized schema. For Bank C, `day_of_week` encodes day-index (0/1) not real weekdays — documented as realistic non-IID feature semantics mismatch.

All original datasets in `data/sparkov/`, `data/paysim/`, `data/creditcard/` remain **untouched**.